In [1]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import itertools

import time

## DATA

In [2]:
# Main Data Folder
DataFolder = '../DATA/'

#### data structure

In [3]:
print('These are the features stored in each .h5:')
    
print('\n bbaa \n')
fname = DataFolder + "backgrounds/bbaa_cuts2.h5"
data_stored = {}

with h5py.File(fname, "r") as f:
    for k in f.keys():
        data = f[k]
        data_stored[k] = f[k][:]
        print(f"key: {k:12s} shape = {data.shape}, dtype = {data.dtype}")

These are the features stored in each .h5:

 bbaa 

key: MET_phi      shape = (129262,), dtype = object
key: MET_pt       shape = (129262,), dtype = object
key: btag_eta     shape = (129262,), dtype = object
key: btag_phi     shape = (129262,), dtype = object
key: btag_pt      shape = (129262,), dtype = object
key: jet_eta      shape = (129262,), dtype = object
key: jet_phi      shape = (129262,), dtype = object
key: jet_pt       shape = (129262,), dtype = object
key: photon_eta   shape = (129262,), dtype = object
key: photon_phi   shape = (129262,), dtype = object
key: photon_pt    shape = (129262,), dtype = object


In [4]:
print(' Each index contains the info of each event:')
print('Number of events: ', len(data_stored['jet_pt']))


print('\n If an event has more than 1 particle of the same type (muliple jets, multiple photons, ...), they are stored in the same variable')
for i in range(5):
    print('Event #:', i, '  number of jets:', len(data_stored['jet_pt'][i]),  '   p_T [GeV]: ', data_stored['jet_pt'][i])

print()
for i in range(5):
    print('Event #:', i, '  number of b-tag jets:', len(data_stored['btag_pt'][i]),  '   p_T [GeV]: ', data_stored['btag_pt'][i])

print()
for i in range(5):
    print('Event #:', i, '  number of photons:', len(data_stored['photon_pt'][i]),  '   p_T [GeV]: ', data_stored['photon_pt'][i])

 Each index contains the info of each event:
Number of events:  129262

 If an event has more than 1 particle of the same type (muliple jets, multiple photons, ...), they are stored in the same variable
Event #: 0   number of jets: 1    p_T [GeV]:  [168.22]
Event #: 1   number of jets: 2    p_T [GeV]:  [283.5   41.98]
Event #: 2   number of jets: 3    p_T [GeV]:  [153.8   77.83  75.35]
Event #: 3   number of jets: 0    p_T [GeV]:  []
Event #: 4   number of jets: 1    p_T [GeV]:  [33.12]

Event #: 0   number of b-tag jets: 2    p_T [GeV]:  [61.24 30.72]
Event #: 1   number of b-tag jets: 2    p_T [GeV]:  [125.15  64.33]
Event #: 2   number of b-tag jets: 2    p_T [GeV]:  [44.42 43.88]
Event #: 3   number of b-tag jets: 2    p_T [GeV]:  [108.46  54.08]
Event #: 4   number of b-tag jets: 2    p_T [GeV]:  [179.59  40.58]

Event #: 0   number of photons: 2    p_T [GeV]:  [137.77  95.61]
Event #: 1   number of photons: 2    p_T [GeV]:  [262.94  42.8 ]
Event #: 2   number of photons: 2    p_T

In [5]:
# to save memory
del data_stored

### HIGH LEVEL FEATURES

In [6]:
def delta_phi(p1, p2):
    """ phi diff in [-π, π]."""
    
    phi1 = p1[:,2]
    phi2 = p2[:,2]
    
    dphi = phi1 - phi2
    dphi = (dphi + np.pi) % (2 * np.pi) - np.pi
    return dphi

def delta_R(p1, p2):
    """ΔR given (eta, phi)."""
    
    eta1 = p1[:,1]
    phi1 = p1[:,2]
    
    eta2 = p2[:,1]
    phi2 = p2[:,2]
    
    d_eta = eta1 - eta2
    d_phi = delta_phi(p1, p2)
    return np.sqrt(d_eta**2 + d_phi**2)

def eta_from_4vec(px, py, pz, E):
    # avoid division problems
    if E != abs(pz):
        return 0.5 * np.log((E + pz) / (E - pz))
    else:
        return 0.0


def inv_mass(p1, p2):
    """invariant mass (massless particles approx)."""
    pt1 = p1[:,0]
    eta1 = p1[:,1]
    phi1 = p1[:,2]
    
    pt2 = p2[:,0]
    eta2 = p2[:,1]
    phi2 = p2[:,2]
    
    
    delta_eta = eta1 - eta2
    delta_phi = np.mod(phi1 - phi2 + np.pi, 2 * np.pi) - np.pi  # [-π, π]
    
    cosh_delta_eta = np.cosh(delta_eta)
    cos_delta_phi = np.cos(delta_phi)
    
    m2 = 2 * pt1 * pt2 * (cosh_delta_eta - cos_delta_phi)
    m2 = np.maximum(m2, 0.0)  # avoid negative roots
    return np.sqrt(m2)


def inv_mass_4vec(px, py, pz, E):
    """invariant mass (massless particles approx). 
    But for 4 vectors"""
    m2 = E**2 - (px**2 + py**2 + pz**2)
    m2 = np.maximum(m2, 0.0)
    return np.sqrt(m2)


def trans_mass(p1, p2):
    """transverse mass between two objects (pt, phi)."""
    pt1 = p1[:,0]
    phi1 = p1[:,2]
    
    pt2 = p2[:,0]
    phi2 = p2[:,2]
    
    delta_phi = np.mod(phi1 - phi2 + np.pi, 2 * np.pi) - np.pi  # [-π, π]
    return np.sqrt(2 * pt1 * pt2 * (1 - np.cos(delta_phi)))


def sT(tau, b1, b2, jet1, jet2):
    """
    the scalar sum of the transverse momenta of the tau lepton and the two leading jets.
    """
    tau_pt = tau[:, 0]
    
    # Stack pt of b1, b2, jet1, jet2 => shape (N, 4)
    others_pt = np.stack([b1[:, 0], b2[:, 0], jet1[:, 0], jet2[:, 0]], axis=1)
    
    # Obtain the 2 higher pTs per row
    top2_pt = np.partition(others_pt, -2, axis=1)[:, -2:] 
    
    # Sum the pTs: tau_pt + top2_pt.sum(axis=1)
    total_pt = tau_pt + np.sum(top2_pt, axis=1)
    
    return total_pt


def pT12(p1, p2):
    """
    transverse momenta of the p1 p2 system
    """
    px1 = p1[:, 0] * np.cos(p1[:, 1])
    px2 = p2[:, 0] * np.cos(p2[:, 1])

    px12 = px1 + px2

    py1 = p1[:, 0] * np.sin(p1[:, 1])
    py2 = p2[:, 0] * np.sin(p2[:, 1])

    py12 = py1 + py2

    return np.sqrt( px12**2 + py12**2 )


def costhetaCS(p1, p2):
    """
    cosine of the angle between p1 and p2 in Collins-Soper frame
    """
    dR12 = p1[:,2] - p2[:,2]
    p_T_12 = pT12(p1, p2)
    m12 = inv_mass(p1, p2)

    return ( np.sinh(dR12) / np.sqrt( 1 + (p_T_12/m12)**2 ) ) * (2 * p1[:,0] * p2[:,0] / (m12**2) )



def higgs_four_vector(p1, p2):
    """
    reconstruct the four vector of the mother particle (assuming it decays to p1 and p2)
    """
    pt1, eta1, phi1 = p1.T
    pt2, eta2, phi2 = p2.T

    px = pt1*np.cos(phi1) + pt2*np.cos(phi2)
    py = pt1*np.sin(phi1) + pt2*np.sin(phi2)
    pz = pt1*np.sinh(eta1) + pt2*np.sinh(eta2)
    E  = pt1*np.cosh(eta1) + pt2*np.cosh(eta2)

    return px, py, pz, E

In [7]:

MH = 125.0

# TRY ALL POSSIBLE COMBINATIONS ADN KEEP THE ONE THAT IS CLOSEST TO THE HIGGS MASS
def best_pair_mass(particles):
    """
    particles: array (N,3) with [pt, eta, phi]
    returns: indices (i,j), mass
    """
    best = None
    best_dm = 1e9

    for i, j in itertools.combinations(range(len(particles)), 2):
        m = inv_mass(particles[i:i+1], particles[j:j+1])[0]
        dm = abs(m - MH)
        if dm < best_dm:
            best_dm = dm
            best = (i, j, m)

    return best  # (i, j, m)

### Function to build the feature set

In [8]:
def build_features_event(photon_pt, photon_eta, photon_phi,
                          btag_pt, btag_eta, btag_phi,
                          jet_pt, jet_eta, jet_phi,
                          MET_pt, MET_phi):
    feats = []

    # multiplicidades
    feats.append(len(photon_pt))
    feats.append(len(jet_pt))
    feats.append(len(btag_pt))

    # MET
    feats.append(MET_pt[0])
    feats.append(MET_phi[0])

    # safety
    if len(photon_pt) < 2 or len(btag_pt) < 2:
        return None

    photons = np.column_stack([photon_pt, photon_eta, photon_phi])
    bjets   = np.column_stack([btag_pt, btag_eta, btag_phi])

    ip, jp, m_phopho = best_pair_mass(photons)
    ib, jb, m_bb = best_pair_mass(bjets)

    g1, g2 = photons[ip], photons[jp]
    b1, b2 = bjets[ib], bjets[jb]

    # pTs
    feats.extend([
        g1[0], g1[1], g1[2],
        g2[0], g2[1], g2[2],
        b1[0], b1[1], b1[2], 
        b2[0], b2[1], b2[2]
    ])

    # masses
    # --- Higgs bb ---
    px_bb, py_bb, pz_bb, E_bb = higgs_four_vector(b1, b2)
    eta_bb = eta_from_4vec(px_bb, py_bb, pz_bb, E_bb)
    phi_bb = np.arctan2(py_bb, px_bb)

    # --- Higgs γγ ---
    px_gg, py_gg, pz_gg, E_gg = higgs_four_vector(g1, g2)
    eta_gg = eta_from_4vec(px_gg, py_gg, pz_gg, E_gg)
    phi_gg = np.arctan2(py_gg, px_gg)

    # --- HH system ---
    px_HH = px_bb + px_gg
    py_HH = py_bb + py_gg
    pz_HH = pz_bb + pz_gg
    E_HH  = E_bb  + E_gg
    
    m_HH = inv_mass_4vec(px_HH, py_HH, pz_HH, E_HH)

    feats.append(m_HH)
    feats.append(m_phopho)
    feats.append(m_bb)

    # pT of systems
    feats.append(np.sqrt(px_HH**2 + py_HH**2))
    feats.append(pT12(g1[None,:], g2[None,:])[0])
    feats.append(pT12(b1[None,:], b2[None,:])[0])
    

    # angular
    delta_eta = eta_bb - eta_gg
    delta_phi = np.mod(phi_bb - phi_gg + np.pi, 2*np.pi) - np.pi

    deltaR_HH = np.sqrt(delta_eta**2 + delta_phi**2)

    feats.append(deltaR_HH)
    feats.append(delta_R(g1[None,:], g2[None,:])[0])
    feats.append(delta_R(b1[None,:], b2[None,:])[0])

    return feats


feats_label = ["num_photons", "num_jets", "num_btag",
               "MET_pT", "MET_phi",
               "photon1_pT", "photon1_eta", "photon1_phi",
               "photon2_pT", "photon2_eta", "photon2_phi",
               "btag1_pt", "btag1_eta", "btag1_phi",
               "btag2_pt", "btag2_eta", "btag2_phi",
               "m_hh", "m_phopho", "m_bb",
               "pT_hh", "pT_phopho", "pT_bb",
               "dR_hh", "dR_phopho", "dR_bb"]

## SAVE THE BACKGROUND FEATURES

In [9]:
CHs_selected = ["bbaa"]


for CH_sel in CHs_selected:
    
    start_time = time.time()

    
    print(CH_sel)

        
    fname = DataFolder + "backgrounds/" + CH_sel + "_cuts2.h5"
        
    X_aux1 = []
    
    with h5py.File(fname, "r") as f:
        # Load everything first (RAM should be ok)
        gam_pt = f["photon_pt"][:]
        gam_eta = f["photon_eta"][:]
        gam_phi = f["photon_phi"][:]
        btag_pt = f["btag_pt"][:]
        btag_eta = f["btag_eta"][:]
        btag_phi = f["btag_phi"][:]
        jet_pt = f["jet_pt"][:]
        jet_eta = f["jet_eta"][:]
        jet_phi = f["jet_phi"][:]
        MET_pt = f["MET_pt"][:]
        MET_phi = f["MET_phi"][:]
    
    
        for i in range(len(gam_pt)):
            feats = build_features_event(
                gam_pt[i],
                gam_eta[i],
                gam_phi[i],
                btag_pt[i],
                btag_eta[i],
                btag_phi[i],
                jet_pt[i],
                jet_eta[i],
                jet_phi[i],
                MET_pt[i],
                MET_phi[i],
            )
    
            if feats is None:
                continue
    
            X_aux1.append(feats)
    
    X_aux2 = np.asarray(X_aux1, dtype=np.float32)
    
    
    print('# of events available:          ', len(X_aux2))
    
    end_time = time.time()
    duracion_h5 = end_time - start_time
    print(f"Time processing the .h5:         {duracion_h5:.2f} s")
    
    
    
    
    # Create the DataFrame
    df = pd.DataFrame(X_aux2, columns=feats_label)
    
    # Save to CSV
    output_folder = DataFolder + "backgrounds/data_df/"
    
    filename = "df_" + CH_sel + ".csv"
    df.to_csv(output_folder + filename, index=False)
    
    print(f"File with features saved:     {filename}")

    print("\n--------------------------------------------------\n")

bbaa
# of events available:           129262
Time processing the .h5:         16.32 s
File with features saved:     df_bbaa.csv

--------------------------------------------------



In [9]:
CHs_selected = ["tth", "zh"]


for CH_sel in CHs_selected:
    
    start_time = time.time()

    
    print(CH_sel)

        
    fname = DataFolder + "backgrounds/" + CH_sel + "_cuts2.h5"
        
    X_aux1 = []
    
    with h5py.File(fname, "r") as f:
        # Load everything first (RAM should be ok)
        gam_pt = f["photon_pt"][:]
        gam_eta = f["photon_eta"][:]
        gam_phi = f["photon_phi"][:]
        btag_pt = f["btag_pt"][:]
        btag_eta = f["btag_eta"][:]
        btag_phi = f["btag_phi"][:]
        jet_pt = f["jet_pt"][:]
        jet_eta = f["jet_eta"][:]
        jet_phi = f["jet_phi"][:]
        MET_pt = f["MET_pt"][:]
        MET_phi = f["MET_phi"][:]
    
    
        for i in range(len(gam_pt)):
            feats = build_features_event(
                gam_pt[i],
                gam_eta[i],
                gam_phi[i],
                btag_pt[i],
                btag_eta[i],
                btag_phi[i],
                jet_pt[i],
                jet_eta[i],
                jet_phi[i],
                MET_pt[i],
                MET_phi[i],
            )
    
            if feats is None:
                continue
    
            X_aux1.append(feats)
    
    X_aux2 = np.asarray(X_aux1, dtype=np.float32)
    
    
    print('# of events available:          ', len(X_aux2))
    
    end_time = time.time()
    duracion_h5 = end_time - start_time
    print(f"Time processing the .h5:         {duracion_h5:.2f} s")
    
    
    
    
    # Create the DataFrame
    df = pd.DataFrame(X_aux2, columns=feats_label)
    
    # Save to CSV
    output_folder = DataFolder + "backgrounds/data_df/"
    
    filename = "df_" + CH_sel + ".csv"
    df.to_csv(output_folder + filename, index=False)
    
    print(f"File with features saved:     {filename}")

    print("\n--------------------------------------------------\n")

bbaa
# of events available:           67056
Time processing the .h5:         7.97 s
File with features saved:     df_bbaa.csv

--------------------------------------------------

tth
# of events available:           154321
Time processing the .h5:         18.36 s
File with features saved:     df_tth.csv

--------------------------------------------------

zh
# of events available:           57657
Time processing the .h5:         6.88 s
File with features saved:     df_zh.csv

--------------------------------------------------



## SAVE THE SM HIGGS FEATURES

In [9]:
BPs_selected = ["BP0"]


for BP_sel in BPs_selected:
    
    start_time = time.time()

    
    print(BP_sel)

        
    fname = DataFolder + "SMhiggs/" + BP_sel + "_cuts2.h5"
        
    X_aux1 = []
    
    with h5py.File(fname, "r") as f:
        # Load everything first (RAM should be ok)
        gam_pt = f["photon_pt"][:]
        gam_eta = f["photon_eta"][:]
        gam_phi = f["photon_phi"][:]
        btag_pt = f["btag_pt"][:]
        btag_eta = f["btag_eta"][:]
        btag_phi = f["btag_phi"][:]
        jet_pt = f["jet_pt"][:]
        jet_eta = f["jet_eta"][:]
        jet_phi = f["jet_phi"][:]
        MET_pt = f["MET_pt"][:]
        MET_phi = f["MET_phi"][:]
    
    
        for i in range(len(gam_pt)):
            feats = build_features_event(
                gam_pt[i],
                gam_eta[i],
                gam_phi[i],
                btag_pt[i],
                btag_eta[i],
                btag_phi[i],
                jet_pt[i],
                jet_eta[i],
                jet_phi[i],
                MET_pt[i],
                MET_phi[i],
            )
    
            if feats is None:
                continue
    
            X_aux1.append(feats)
    
    X_aux2 = np.asarray(X_aux1, dtype=np.float32)
    
    
    print('# of events available:          ', len(X_aux2))
    
    end_time = time.time()
    duracion_h5 = end_time - start_time
    print(f"Time processing the .h5:         {duracion_h5:.2f} s")
    
    
    
    
    # Create the DataFrame
    df = pd.DataFrame(X_aux2, columns=feats_label)
    
    # Save to CSV
    output_folder = DataFolder + "SMhiggs/data_df/"
    
    filename = "df_" + BP_sel + ".csv"
    df.to_csv(output_folder + filename, index=False)
    
    print(f"File with features saved:     {filename}")

    print("\n--------------------------------------------------\n")

BP0
# of events available:           35960
Time processing the .h5:         4.04 s
File with features saved:     df_BP0.csv

--------------------------------------------------



## SAVE THE SIGNAL FEATURES

In [10]:
# BP1 IS STORED AS point1
BPs_selected = ["BP1"]

for BP_sel in BPs_selected:
    
    start_time = time.time()

    
    print(BP_sel)

        
    fname = DataFolder + "signal/point1_cuts2.h5"
        
    X_aux1 = []
    
    with h5py.File(fname, "r") as f:
        # Load everything first (RAM should be ok)
        gam_pt = f["photon_pt"][:]
        gam_eta = f["photon_eta"][:]
        gam_phi = f["photon_phi"][:]
        btag_pt = f["btag_pt"][:]
        btag_eta = f["btag_eta"][:]
        btag_phi = f["btag_phi"][:]
        jet_pt = f["jet_pt"][:]
        jet_eta = f["jet_eta"][:]
        jet_phi = f["jet_phi"][:]
        MET_pt = f["MET_pt"][:]
        MET_phi = f["MET_phi"][:]
    
    
        for i in range(len(gam_pt)):
            feats = build_features_event(
                gam_pt[i],
                gam_eta[i],
                gam_phi[i],
                btag_pt[i],
                btag_eta[i],
                btag_phi[i],
                jet_pt[i],
                jet_eta[i],
                jet_phi[i],
                MET_pt[i],
                MET_phi[i],
            )
    
            if feats is None:
                continue
    
            X_aux1.append(feats)
    
    X_aux2 = np.asarray(X_aux1, dtype=np.float32)
    
    
    print('# of events available:          ', len(X_aux2))
    
    end_time = time.time()
    duracion_h5 = end_time - start_time
    print(f"Time processing the .h5:         {duracion_h5:.2f} s")
    
    
    
    
    # Create the DataFrame
    df = pd.DataFrame(X_aux2, columns=feats_label)
    
    # Save to CSV
    output_folder = DataFolder + "signal/data_df/"
    
    filename = "df_" + BP_sel + ".csv"
    df.to_csv(output_folder + filename, index=False)
    
    print(f"File with features saved:     {filename}")

    print("\n--------------------------------------------------\n")

BP1
# of events available:           153551
Time processing the .h5:         17.79 s
File with features saved:     df_BP1.csv

--------------------------------------------------



In [11]:
BPs_selected = ["BP28", "BP62", "BP63",
         "BP219", "BP220", "BP586", "BP635",
         "BP673", "BP684", "BP685", "BP728", "BP754"]


for BP_sel in BPs_selected:
    
    start_time = time.time()

    
    print(BP_sel)

        
    fname = DataFolder + "signal/" + BP_sel + "_cuts2.h5"
        
    X_aux1 = []
    
    with h5py.File(fname, "r") as f:
        # Load everything first (RAM should be ok)
        gam_pt = f["photon_pt"][:]
        gam_eta = f["photon_eta"][:]
        gam_phi = f["photon_phi"][:]
        btag_pt = f["btag_pt"][:]
        btag_eta = f["btag_eta"][:]
        btag_phi = f["btag_phi"][:]
        jet_pt = f["jet_pt"][:]
        jet_eta = f["jet_eta"][:]
        jet_phi = f["jet_phi"][:]
        MET_pt = f["MET_pt"][:]
        MET_phi = f["MET_phi"][:]
    
    
        for i in range(len(gam_pt)):
            feats = build_features_event(
                gam_pt[i],
                gam_eta[i],
                gam_phi[i],
                btag_pt[i],
                btag_eta[i],
                btag_phi[i],
                jet_pt[i],
                jet_eta[i],
                jet_phi[i],
                MET_pt[i],
                MET_phi[i],
            )
    
            if feats is None:
                continue
    
            X_aux1.append(feats)
    
    X_aux2 = np.asarray(X_aux1, dtype=np.float32)
    
    
    print('# of events available:          ', len(X_aux2))
    
    end_time = time.time()
    duracion_h5 = end_time - start_time
    print(f"Time processing the .h5:         {duracion_h5:.2f} s")
    
    
    
    
    # Create the DataFrame
    df = pd.DataFrame(X_aux2, columns=feats_label)
    
    # Save to CSV
    output_folder = DataFolder + "signal/data_df/"
    
    filename = "df_" + BP_sel + ".csv"
    df.to_csv(output_folder + filename, index=False)
    
    print(f"File with features saved:     {filename}")

    print("\n--------------------------------------------------\n")

BP28
# of events available:           7300
Time processing the .h5:         1.76 s
File with features saved:     df_BP28.csv

--------------------------------------------------

BP62
# of events available:           7736
Time processing the .h5:         0.90 s
File with features saved:     df_BP62.csv

--------------------------------------------------

BP63
# of events available:           8062
Time processing the .h5:         0.94 s
File with features saved:     df_BP63.csv

--------------------------------------------------

BP219
# of events available:           7590
Time processing the .h5:         0.86 s
File with features saved:     df_BP219.csv

--------------------------------------------------

BP220
# of events available:           11096
Time processing the .h5:         1.22 s
File with features saved:     df_BP220.csv

--------------------------------------------------

BP586
# of events available:           11057
Time processing the .h5:         1.31 s
File with features s

In [12]:
BPs_selected = ["BP754", "BP758", "BP763"]


for BP_sel in BPs_selected:
    
    start_time = time.time()

    
    print(BP_sel)

        
    fname = DataFolder + "signal/" + BP_sel + "_cuts2.h5"
        
    X_aux1 = []
    
    with h5py.File(fname, "r") as f:
        # Load everything first (RAM should be ok)
        gam_pt = f["photon_pt"][:]
        gam_eta = f["photon_eta"][:]
        gam_phi = f["photon_phi"][:]
        btag_pt = f["btag_pt"][:]
        btag_eta = f["btag_eta"][:]
        btag_phi = f["btag_phi"][:]
        jet_pt = f["jet_pt"][:]
        jet_eta = f["jet_eta"][:]
        jet_phi = f["jet_phi"][:]
        MET_pt = f["MET_pt"][:]
        MET_phi = f["MET_phi"][:]
    
    
        for i in range(len(gam_pt)):
            feats = build_features_event(
                gam_pt[i],
                gam_eta[i],
                gam_phi[i],
                btag_pt[i],
                btag_eta[i],
                btag_phi[i],
                jet_pt[i],
                jet_eta[i],
                jet_phi[i],
                MET_pt[i],
                MET_phi[i],
            )
    
            if feats is None:
                continue
    
            X_aux1.append(feats)
    
    X_aux2 = np.asarray(X_aux1, dtype=np.float32)
    
    
    print('# of events available:          ', len(X_aux2))
    
    end_time = time.time()
    duracion_h5 = end_time - start_time
    print(f"Time processing the .h5:         {duracion_h5:.2f} s")
    
    
    
    
    # Create the DataFrame
    df = pd.DataFrame(X_aux2, columns=feats_label)
    
    # Save to CSV
    output_folder = DataFolder + "signal/data_df/"
    
    filename = "df_" + BP_sel + ".csv"
    df.to_csv(output_folder + filename, index=False)
    
    print(f"File with features saved:     {filename}")

    print("\n--------------------------------------------------\n")

BP754
# of events available:           7377
Time processing the .h5:         1.02 s
File with features saved:     df_BP754.csv

--------------------------------------------------

BP758
# of events available:           7423
Time processing the .h5:         0.95 s
File with features saved:     df_BP758.csv

--------------------------------------------------

BP763
# of events available:           6727
Time processing the .h5:         1.05 s
File with features saved:     df_BP763.csv

--------------------------------------------------



In [9]:
BPs_selected = ["BP765"]


for BP_sel in BPs_selected:
    
    start_time = time.time()

    
    print(BP_sel)

        
    fname = DataFolder + "signal/" + BP_sel + "_cuts2.h5"
        
    X_aux1 = []
    
    with h5py.File(fname, "r") as f:
        # Load everything first (RAM should be ok)
        gam_pt = f["photon_pt"][:]
        gam_eta = f["photon_eta"][:]
        gam_phi = f["photon_phi"][:]
        btag_pt = f["btag_pt"][:]
        btag_eta = f["btag_eta"][:]
        btag_phi = f["btag_phi"][:]
        jet_pt = f["jet_pt"][:]
        jet_eta = f["jet_eta"][:]
        jet_phi = f["jet_phi"][:]
        MET_pt = f["MET_pt"][:]
        MET_phi = f["MET_phi"][:]
    
    
        for i in range(len(gam_pt)):
            feats = build_features_event(
                gam_pt[i],
                gam_eta[i],
                gam_phi[i],
                btag_pt[i],
                btag_eta[i],
                btag_phi[i],
                jet_pt[i],
                jet_eta[i],
                jet_phi[i],
                MET_pt[i],
                MET_phi[i],
            )
    
            if feats is None:
                continue
    
            X_aux1.append(feats)
    
    X_aux2 = np.asarray(X_aux1, dtype=np.float32)
    
    
    print('# of events available:          ', len(X_aux2))
    
    end_time = time.time()
    duracion_h5 = end_time - start_time
    print(f"Time processing the .h5:         {duracion_h5:.2f} s")
    
    
    
    
    # Create the DataFrame
    df = pd.DataFrame(X_aux2, columns=feats_label)
    
    # Save to CSV
    output_folder = DataFolder + "signal/data_df/"
    
    filename = "df_" + BP_sel + ".csv"
    df.to_csv(output_folder + filename, index=False)
    
    print(f"File with features saved:     {filename}")

    print("\n--------------------------------------------------\n")

BP765
# of events available:           7005
Time processing the .h5:         0.86 s
File with features saved:     df_BP765.csv

--------------------------------------------------



In [9]:
BPs_selected = ["BP787"]


for BP_sel in BPs_selected:
    
    start_time = time.time()

    
    print(BP_sel)

        
    fname = DataFolder + "signal/" + BP_sel + "_cuts2.h5"
        
    X_aux1 = []
    
    with h5py.File(fname, "r") as f:
        # Load everything first (RAM should be ok)
        gam_pt = f["photon_pt"][:]
        gam_eta = f["photon_eta"][:]
        gam_phi = f["photon_phi"][:]
        btag_pt = f["btag_pt"][:]
        btag_eta = f["btag_eta"][:]
        btag_phi = f["btag_phi"][:]
        jet_pt = f["jet_pt"][:]
        jet_eta = f["jet_eta"][:]
        jet_phi = f["jet_phi"][:]
        MET_pt = f["MET_pt"][:]
        MET_phi = f["MET_phi"][:]
    
    
        for i in range(len(gam_pt)):
            feats = build_features_event(
                gam_pt[i],
                gam_eta[i],
                gam_phi[i],
                btag_pt[i],
                btag_eta[i],
                btag_phi[i],
                jet_pt[i],
                jet_eta[i],
                jet_phi[i],
                MET_pt[i],
                MET_phi[i],
            )
    
            if feats is None:
                continue
    
            X_aux1.append(feats)
    
    X_aux2 = np.asarray(X_aux1, dtype=np.float32)
    
    
    print('# of events available:          ', len(X_aux2))
    
    end_time = time.time()
    duracion_h5 = end_time - start_time
    print(f"Time processing the .h5:         {duracion_h5:.2f} s")
    
    
    
    
    # Create the DataFrame
    df = pd.DataFrame(X_aux2, columns=feats_label)
    
    # Save to CSV
    output_folder = DataFolder + "signal/data_df/"
    
    filename = "df_" + BP_sel + ".csv"
    df.to_csv(output_folder + filename, index=False)
    
    print(f"File with features saved:     {filename}")

    print("\n--------------------------------------------------\n")

BP787
# of events available:           7734
Time processing the .h5:         0.91 s
File with features saved:     df_BP787.csv

--------------------------------------------------



In [9]:
BPs_selected = ["BP732", "BP850", "BP855"]


for BP_sel in BPs_selected:
    
    start_time = time.time()

    
    print(BP_sel)

        
    fname = DataFolder + "signal/" + BP_sel + "_cuts2.h5"
        
    X_aux1 = []
    
    with h5py.File(fname, "r") as f:
        # Load everything first (RAM should be ok)
        gam_pt = f["photon_pt"][:]
        gam_eta = f["photon_eta"][:]
        gam_phi = f["photon_phi"][:]
        btag_pt = f["btag_pt"][:]
        btag_eta = f["btag_eta"][:]
        btag_phi = f["btag_phi"][:]
        jet_pt = f["jet_pt"][:]
        jet_eta = f["jet_eta"][:]
        jet_phi = f["jet_phi"][:]
        MET_pt = f["MET_pt"][:]
        MET_phi = f["MET_phi"][:]
    
    
        for i in range(len(gam_pt)):
            feats = build_features_event(
                gam_pt[i],
                gam_eta[i],
                gam_phi[i],
                btag_pt[i],
                btag_eta[i],
                btag_phi[i],
                jet_pt[i],
                jet_eta[i],
                jet_phi[i],
                MET_pt[i],
                MET_phi[i],
            )
    
            if feats is None:
                continue
    
            X_aux1.append(feats)
    
    X_aux2 = np.asarray(X_aux1, dtype=np.float32)
    
    
    print('# of events available:          ', len(X_aux2))
    
    end_time = time.time()
    duracion_h5 = end_time - start_time
    print(f"Time processing the .h5:         {duracion_h5:.2f} s")
    
    
    
    
    # Create the DataFrame
    df = pd.DataFrame(X_aux2, columns=feats_label)
    
    # Save to CSV
    output_folder = DataFolder + "signal/data_df/"
    
    filename = "df_" + BP_sel + ".csv"
    df.to_csv(output_folder + filename, index=False)
    
    print(f"File with features saved:     {filename}")

    print("\n--------------------------------------------------\n")

BP732
# of events available:           6848
Time processing the .h5:         0.81 s
File with features saved:     df_BP732.csv

--------------------------------------------------

BP850
# of events available:           7324
Time processing the .h5:         0.85 s
File with features saved:     df_BP850.csv

--------------------------------------------------

BP855
# of events available:           7321
Time processing the .h5:         0.83 s
File with features saved:     df_BP855.csv

--------------------------------------------------



In [9]:
BPs_selected = ["BP872"]


for BP_sel in BPs_selected:
    
    start_time = time.time()

    
    print(BP_sel)

        
    fname = DataFolder + "signal/" + BP_sel + "_cuts2.h5"
        
    X_aux1 = []
    
    with h5py.File(fname, "r") as f:
        # Load everything first (RAM should be ok)
        gam_pt = f["photon_pt"][:]
        gam_eta = f["photon_eta"][:]
        gam_phi = f["photon_phi"][:]
        btag_pt = f["btag_pt"][:]
        btag_eta = f["btag_eta"][:]
        btag_phi = f["btag_phi"][:]
        jet_pt = f["jet_pt"][:]
        jet_eta = f["jet_eta"][:]
        jet_phi = f["jet_phi"][:]
        MET_pt = f["MET_pt"][:]
        MET_phi = f["MET_phi"][:]
    
    
        for i in range(len(gam_pt)):
            feats = build_features_event(
                gam_pt[i],
                gam_eta[i],
                gam_phi[i],
                btag_pt[i],
                btag_eta[i],
                btag_phi[i],
                jet_pt[i],
                jet_eta[i],
                jet_phi[i],
                MET_pt[i],
                MET_phi[i],
            )
    
            if feats is None:
                continue
    
            X_aux1.append(feats)
    
    X_aux2 = np.asarray(X_aux1, dtype=np.float32)
    
    
    print('# of events available:          ', len(X_aux2))
    
    end_time = time.time()
    duracion_h5 = end_time - start_time
    print(f"Time processing the .h5:         {duracion_h5:.2f} s")
    
    
    
    
    # Create the DataFrame
    df = pd.DataFrame(X_aux2, columns=feats_label)
    
    # Save to CSV
    output_folder = DataFolder + "signal/data_df/"
    
    filename = "df_" + BP_sel + ".csv"
    df.to_csv(output_folder + filename, index=False)
    
    print(f"File with features saved:     {filename}")

    print("\n--------------------------------------------------\n")

BP872
# of events available:           7114
Time processing the .h5:         0.86 s
File with features saved:     df_BP872.csv

--------------------------------------------------



In [12]:
BPs_selected = ["BP752"]


for BP_sel in BPs_selected:
    
    start_time = time.time()

    
    print(BP_sel)

        
    fname = DataFolder + "signal/" + BP_sel + "_cuts2.h5"
        
    X_aux1 = []
    
    with h5py.File(fname, "r") as f:
        # Load everything first (RAM should be ok)
        gam_pt = f["photon_pt"][:]
        gam_eta = f["photon_eta"][:]
        gam_phi = f["photon_phi"][:]
        btag_pt = f["btag_pt"][:]
        btag_eta = f["btag_eta"][:]
        btag_phi = f["btag_phi"][:]
        jet_pt = f["jet_pt"][:]
        jet_eta = f["jet_eta"][:]
        jet_phi = f["jet_phi"][:]
        MET_pt = f["MET_pt"][:]
        MET_phi = f["MET_phi"][:]
    
    
        for i in range(len(gam_pt)):
            feats = build_features_event(
                gam_pt[i],
                gam_eta[i],
                gam_phi[i],
                btag_pt[i],
                btag_eta[i],
                btag_phi[i],
                jet_pt[i],
                jet_eta[i],
                jet_phi[i],
                MET_pt[i],
                MET_phi[i],
            )
    
            if feats is None:
                continue
    
            X_aux1.append(feats)
    
    X_aux2 = np.asarray(X_aux1, dtype=np.float32)
    
    
    print('# of events available:          ', len(X_aux2))
    
    end_time = time.time()
    duracion_h5 = end_time - start_time
    print(f"Time processing the .h5:         {duracion_h5:.2f} s")
    
    
    
    
    # Create the DataFrame
    df = pd.DataFrame(X_aux2, columns=feats_label)
    
    # Save to CSV
    output_folder = DataFolder + "signal/data_df/"
    
    filename = "df_" + BP_sel + ".csv"
    df.to_csv(output_folder + filename, index=False)
    
    print(f"File with features saved:     {filename}")

    print("\n--------------------------------------------------\n")

BP752
# of events available:           6785
Time processing the .h5:         0.84 s
File with features saved:     df_BP752.csv

--------------------------------------------------



In [11]:
BPs_selected = ["BP826", "BP841", "BP875", "BP881"]


for BP_sel in BPs_selected:
    
    start_time = time.time()

    
    print(BP_sel)

        
    fname = DataFolder + "signal/" + BP_sel + "_cuts2.h5"
        
    X_aux1 = []
    
    with h5py.File(fname, "r") as f:
        # Load everything first (RAM should be ok)
        gam_pt = f["photon_pt"][:]
        gam_eta = f["photon_eta"][:]
        gam_phi = f["photon_phi"][:]
        btag_pt = f["btag_pt"][:]
        btag_eta = f["btag_eta"][:]
        btag_phi = f["btag_phi"][:]
        jet_pt = f["jet_pt"][:]
        jet_eta = f["jet_eta"][:]
        jet_phi = f["jet_phi"][:]
        MET_pt = f["MET_pt"][:]
        MET_phi = f["MET_phi"][:]
    
    
        for i in range(len(gam_pt)):
            feats = build_features_event(
                gam_pt[i],
                gam_eta[i],
                gam_phi[i],
                btag_pt[i],
                btag_eta[i],
                btag_phi[i],
                jet_pt[i],
                jet_eta[i],
                jet_phi[i],
                MET_pt[i],
                MET_phi[i],
            )
    
            if feats is None:
                continue
    
            X_aux1.append(feats)
    
    X_aux2 = np.asarray(X_aux1, dtype=np.float32)
    
    
    print('# of events available:          ', len(X_aux2))
    
    end_time = time.time()
    duracion_h5 = end_time - start_time
    print(f"Time processing the .h5:         {duracion_h5:.2f} s")
    
    
    
    
    # Create the DataFrame
    df = pd.DataFrame(X_aux2, columns=feats_label)
    
    # Save to CSV
    output_folder = DataFolder + "signal/data_df/"
    
    filename = "df_" + BP_sel + ".csv"
    df.to_csv(output_folder + filename, index=False)
    
    print(f"File with features saved:     {filename}")

    print("\n--------------------------------------------------\n")

BP752
# of events available:           13570
Time processing the .h5:         1.60 s
File with features saved:     df_BP752.csv

--------------------------------------------------

BP826
# of events available:           7178
Time processing the .h5:         0.85 s
File with features saved:     df_BP826.csv

--------------------------------------------------

BP841
# of events available:           6941
Time processing the .h5:         0.77 s
File with features saved:     df_BP841.csv

--------------------------------------------------

BP875
# of events available:           7201
Time processing the .h5:         0.79 s
File with features saved:     df_BP875.csv

--------------------------------------------------

BP881
# of events available:           7209
Time processing the .h5:         0.79 s
File with features saved:     df_BP881.csv

--------------------------------------------------



In [9]:
BPs_selected = ["BP1017", "BP1035"]


for BP_sel in BPs_selected:
    
    start_time = time.time()

    
    print(BP_sel)

        
    fname = DataFolder + "signal/" + BP_sel + "_cuts2.h5"
        
    X_aux1 = []
    
    with h5py.File(fname, "r") as f:
        # Load everything first (RAM should be ok)
        gam_pt = f["photon_pt"][:]
        gam_eta = f["photon_eta"][:]
        gam_phi = f["photon_phi"][:]
        btag_pt = f["btag_pt"][:]
        btag_eta = f["btag_eta"][:]
        btag_phi = f["btag_phi"][:]
        jet_pt = f["jet_pt"][:]
        jet_eta = f["jet_eta"][:]
        jet_phi = f["jet_phi"][:]
        MET_pt = f["MET_pt"][:]
        MET_phi = f["MET_phi"][:]
    
    
        for i in range(len(gam_pt)):
            feats = build_features_event(
                gam_pt[i],
                gam_eta[i],
                gam_phi[i],
                btag_pt[i],
                btag_eta[i],
                btag_phi[i],
                jet_pt[i],
                jet_eta[i],
                jet_phi[i],
                MET_pt[i],
                MET_phi[i],
            )
    
            if feats is None:
                continue
    
            X_aux1.append(feats)
    
    X_aux2 = np.asarray(X_aux1, dtype=np.float32)
    
    
    print('# of events available:          ', len(X_aux2))
    
    end_time = time.time()
    duracion_h5 = end_time - start_time
    print(f"Time processing the .h5:         {duracion_h5:.2f} s")
    
    
    
    
    # Create the DataFrame
    df = pd.DataFrame(X_aux2, columns=feats_label)
    
    # Save to CSV
    output_folder = DataFolder + "signal/data_df/"
    
    filename = "df_" + BP_sel + ".csv"
    df.to_csv(output_folder + filename, index=False)
    
    print(f"File with features saved:     {filename}")

    print("\n--------------------------------------------------\n")

BP1017
# of events available:           7060
Time processing the .h5:         0.87 s
File with features saved:     df_BP1017.csv

--------------------------------------------------

BP1035
# of events available:           7039
Time processing the .h5:         1.01 s
File with features saved:     df_BP1035.csv

--------------------------------------------------



In [9]:
BPs_selected = ["BP791", "BP1036", "BP1087", "BP1103", "BP1104"]


for BP_sel in BPs_selected:
    
    start_time = time.time()

    
    print(BP_sel)

        
    fname = DataFolder + "signal/" + BP_sel + "_cuts2.h5"
        
    X_aux1 = []
    
    with h5py.File(fname, "r") as f:
        # Load everything first (RAM should be ok)
        gam_pt = f["photon_pt"][:]
        gam_eta = f["photon_eta"][:]
        gam_phi = f["photon_phi"][:]
        btag_pt = f["btag_pt"][:]
        btag_eta = f["btag_eta"][:]
        btag_phi = f["btag_phi"][:]
        jet_pt = f["jet_pt"][:]
        jet_eta = f["jet_eta"][:]
        jet_phi = f["jet_phi"][:]
        MET_pt = f["MET_pt"][:]
        MET_phi = f["MET_phi"][:]
    
    
        for i in range(len(gam_pt)):
            feats = build_features_event(
                gam_pt[i],
                gam_eta[i],
                gam_phi[i],
                btag_pt[i],
                btag_eta[i],
                btag_phi[i],
                jet_pt[i],
                jet_eta[i],
                jet_phi[i],
                MET_pt[i],
                MET_phi[i],
            )
    
            if feats is None:
                continue
    
            X_aux1.append(feats)
    
    X_aux2 = np.asarray(X_aux1, dtype=np.float32)
    
    
    print('# of events available:          ', len(X_aux2))
    
    end_time = time.time()
    duracion_h5 = end_time - start_time
    print(f"Time processing the .h5:         {duracion_h5:.2f} s")
    
    
    
    
    # Create the DataFrame
    df = pd.DataFrame(X_aux2, columns=feats_label)
    
    # Save to CSV
    output_folder = DataFolder + "signal/data_df/"
    
    filename = "df_" + BP_sel + ".csv"
    df.to_csv(output_folder + filename, index=False)
    
    print(f"File with features saved:     {filename}")

    print("\n--------------------------------------------------\n")

BP791
# of events available:           6704
Time processing the .h5:         3.48 s
File with features saved:     df_BP791.csv

--------------------------------------------------

BP1036
# of events available:           7165
Time processing the .h5:         2.92 s
File with features saved:     df_BP1036.csv

--------------------------------------------------

BP1087
# of events available:           7171
Time processing the .h5:         3.77 s
File with features saved:     df_BP1087.csv

--------------------------------------------------

BP1103
# of events available:           7260
Time processing the .h5:         3.66 s
File with features saved:     df_BP1103.csv

--------------------------------------------------

BP1104
# of events available:           7063
Time processing the .h5:         3.80 s
File with features saved:     df_BP1104.csv

--------------------------------------------------



In [11]:
BPs_selected = ["BP658", "BP728"]


for BP_sel in BPs_selected:
    
    start_time = time.time()

    
    print(BP_sel)

        
    fname = DataFolder + "signal/" + BP_sel + "_cuts2.h5"
        
    X_aux1 = []
    
    with h5py.File(fname, "r") as f:
        # Load everything first (RAM should be ok)
        gam_pt = f["photon_pt"][:]
        gam_eta = f["photon_eta"][:]
        gam_phi = f["photon_phi"][:]
        btag_pt = f["btag_pt"][:]
        btag_eta = f["btag_eta"][:]
        btag_phi = f["btag_phi"][:]
        jet_pt = f["jet_pt"][:]
        jet_eta = f["jet_eta"][:]
        jet_phi = f["jet_phi"][:]
        MET_pt = f["MET_pt"][:]
        MET_phi = f["MET_phi"][:]
    
    
        for i in range(len(gam_pt)):
            feats = build_features_event(
                gam_pt[i],
                gam_eta[i],
                gam_phi[i],
                btag_pt[i],
                btag_eta[i],
                btag_phi[i],
                jet_pt[i],
                jet_eta[i],
                jet_phi[i],
                MET_pt[i],
                MET_phi[i],
            )
    
            if feats is None:
                continue
    
            X_aux1.append(feats)
    
    X_aux2 = np.asarray(X_aux1, dtype=np.float32)
    
    
    print('# of events available:          ', len(X_aux2))
    
    end_time = time.time()
    duracion_h5 = end_time - start_time
    print(f"Time processing the .h5:         {duracion_h5:.2f} s")
    
    
    
    
    # Create the DataFrame
    df = pd.DataFrame(X_aux2, columns=feats_label)
    
    # Save to CSV
    output_folder = DataFolder + "signal/data_df/"
    
    filename = "df_" + BP_sel + ".csv"
    df.to_csv(output_folder + filename, index=False)
    
    print(f"File with features saved:     {filename}")

    print("\n--------------------------------------------------\n")

BP658
# of events available:           6823
Time processing the .h5:         0.79 s
File with features saved:     df_BP658.csv

--------------------------------------------------

BP728
# of events available:           10263
Time processing the .h5:         1.15 s
File with features saved:     df_BP728.csv

--------------------------------------------------



In [9]:
BPs_selected = ["BP678", "BP684", "BP685"]


for BP_sel in BPs_selected:
    
    start_time = time.time()

    
    print(BP_sel)

        
    fname = DataFolder + "signal/" + BP_sel + "_cuts2.h5"
        
    X_aux1 = []
    
    with h5py.File(fname, "r") as f:
        # Load everything first (RAM should be ok)
        gam_pt = f["photon_pt"][:]
        gam_eta = f["photon_eta"][:]
        gam_phi = f["photon_phi"][:]
        btag_pt = f["btag_pt"][:]
        btag_eta = f["btag_eta"][:]
        btag_phi = f["btag_phi"][:]
        jet_pt = f["jet_pt"][:]
        jet_eta = f["jet_eta"][:]
        jet_phi = f["jet_phi"][:]
        MET_pt = f["MET_pt"][:]
        MET_phi = f["MET_phi"][:]
    
    
        for i in range(len(gam_pt)):
            feats = build_features_event(
                gam_pt[i],
                gam_eta[i],
                gam_phi[i],
                btag_pt[i],
                btag_eta[i],
                btag_phi[i],
                jet_pt[i],
                jet_eta[i],
                jet_phi[i],
                MET_pt[i],
                MET_phi[i],
            )
    
            if feats is None:
                continue
    
            X_aux1.append(feats)
    
    X_aux2 = np.asarray(X_aux1, dtype=np.float32)
    
    
    print('# of events available:          ', len(X_aux2))
    
    end_time = time.time()
    duracion_h5 = end_time - start_time
    print(f"Time processing the .h5:         {duracion_h5:.2f} s")
    
    
    
    
    # Create the DataFrame
    df = pd.DataFrame(X_aux2, columns=feats_label)
    
    # Save to CSV
    output_folder = DataFolder + "signal/data_df/"
    
    filename = "df_" + BP_sel + ".csv"
    df.to_csv(output_folder + filename, index=False)
    
    print(f"File with features saved:     {filename}")

    print("\n--------------------------------------------------\n")

BP678
# of events available:           6903
Time processing the .h5:         0.89 s
File with features saved:     df_BP678.csv

--------------------------------------------------

BP684
# of events available:           11225
Time processing the .h5:         1.42 s
File with features saved:     df_BP684.csv

--------------------------------------------------

BP685
# of events available:           10343
Time processing the .h5:         1.23 s
File with features saved:     df_BP685.csv

--------------------------------------------------



In [11]:
BPs_selected = ["BP626","BP673", "BP668", "BP996"]


for BP_sel in BPs_selected:
    
    start_time = time.time()

    
    print(BP_sel)

        
    fname = DataFolder + "signal/" + BP_sel + "_cuts2.h5"
        
    X_aux1 = []
    
    with h5py.File(fname, "r") as f:
        # Load everything first (RAM should be ok)
        gam_pt = f["photon_pt"][:]
        gam_eta = f["photon_eta"][:]
        gam_phi = f["photon_phi"][:]
        btag_pt = f["btag_pt"][:]
        btag_eta = f["btag_eta"][:]
        btag_phi = f["btag_phi"][:]
        jet_pt = f["jet_pt"][:]
        jet_eta = f["jet_eta"][:]
        jet_phi = f["jet_phi"][:]
        MET_pt = f["MET_pt"][:]
        MET_phi = f["MET_phi"][:]
    
    
        for i in range(len(gam_pt)):
            feats = build_features_event(
                gam_pt[i],
                gam_eta[i],
                gam_phi[i],
                btag_pt[i],
                btag_eta[i],
                btag_phi[i],
                jet_pt[i],
                jet_eta[i],
                jet_phi[i],
                MET_pt[i],
                MET_phi[i],
            )
    
            if feats is None:
                continue
    
            X_aux1.append(feats)
    
    X_aux2 = np.asarray(X_aux1, dtype=np.float32)
    
    
    print('# of events available:          ', len(X_aux2))
    
    end_time = time.time()
    duracion_h5 = end_time - start_time
    print(f"Time processing the .h5:         {duracion_h5:.2f} s")
    
    
    
    
    # Create the DataFrame
    df = pd.DataFrame(X_aux2, columns=feats_label)
    
    # Save to CSV
    output_folder = DataFolder + "signal/data_df/"
    
    filename = "df_" + BP_sel + ".csv"
    df.to_csv(output_folder + filename, index=False)
    
    print(f"File with features saved:     {filename}")

    print("\n--------------------------------------------------\n")

BP626
# of events available:           7918
Time processing the .h5:         1.18 s
File with features saved:     df_BP626.csv

--------------------------------------------------

BP673
# of events available:           11115
Time processing the .h5:         1.43 s
File with features saved:     df_BP673.csv

--------------------------------------------------

BP668
# of events available:           6766
Time processing the .h5:         0.88 s
File with features saved:     df_BP668.csv

--------------------------------------------------

BP996
# of events available:           7096
Time processing the .h5:         1.01 s
File with features saved:     df_BP996.csv

--------------------------------------------------



In [10]:
BPs_selected = ["BP502"]


for BP_sel in BPs_selected:
    
    start_time = time.time()

    
    print(BP_sel)

        
    fname = DataFolder + "signal/" + BP_sel + "_cuts2.h5"
        
    X_aux1 = []
    
    with h5py.File(fname, "r") as f:
        # Load everything first (RAM should be ok)
        gam_pt = f["photon_pt"][:]
        gam_eta = f["photon_eta"][:]
        gam_phi = f["photon_phi"][:]
        btag_pt = f["btag_pt"][:]
        btag_eta = f["btag_eta"][:]
        btag_phi = f["btag_phi"][:]
        jet_pt = f["jet_pt"][:]
        jet_eta = f["jet_eta"][:]
        jet_phi = f["jet_phi"][:]
        MET_pt = f["MET_pt"][:]
        MET_phi = f["MET_phi"][:]
    
    
        for i in range(len(gam_pt)):
            feats = build_features_event(
                gam_pt[i],
                gam_eta[i],
                gam_phi[i],
                btag_pt[i],
                btag_eta[i],
                btag_phi[i],
                jet_pt[i],
                jet_eta[i],
                jet_phi[i],
                MET_pt[i],
                MET_phi[i],
            )
    
            if feats is None:
                continue
    
            X_aux1.append(feats)
    
    X_aux2 = np.asarray(X_aux1, dtype=np.float32)
    
    
    print('# of events available:          ', len(X_aux2))
    
    end_time = time.time()
    duracion_h5 = end_time - start_time
    print(f"Time processing the .h5:         {duracion_h5:.2f} s")
    
    
    
    
    # Create the DataFrame
    df = pd.DataFrame(X_aux2, columns=feats_label)
    
    # Save to CSV
    output_folder = DataFolder + "signal/data_df/"
    
    filename = "df_" + BP_sel + ".csv"
    df.to_csv(output_folder + filename, index=False)
    
    print(f"File with features saved:     {filename}")

    print("\n--------------------------------------------------\n")

BP502
# of events available:           8914
Time processing the .h5:         1.14 s
File with features saved:     df_BP502.csv

--------------------------------------------------



In [12]:
BPs_selected = ["BP2381"]


for BP_sel in BPs_selected:
    
    start_time = time.time()

    
    print(BP_sel)

        
    fname = DataFolder + "signal/" + BP_sel + "_cuts2.h5"
        
    X_aux1 = []
    
    with h5py.File(fname, "r") as f:
        # Load everything first (RAM should be ok)
        gam_pt = f["photon_pt"][:]
        gam_eta = f["photon_eta"][:]
        gam_phi = f["photon_phi"][:]
        btag_pt = f["btag_pt"][:]
        btag_eta = f["btag_eta"][:]
        btag_phi = f["btag_phi"][:]
        jet_pt = f["jet_pt"][:]
        jet_eta = f["jet_eta"][:]
        jet_phi = f["jet_phi"][:]
        MET_pt = f["MET_pt"][:]
        MET_phi = f["MET_phi"][:]
    
    
        for i in range(len(gam_pt)):
            feats = build_features_event(
                gam_pt[i],
                gam_eta[i],
                gam_phi[i],
                btag_pt[i],
                btag_eta[i],
                btag_phi[i],
                jet_pt[i],
                jet_eta[i],
                jet_phi[i],
                MET_pt[i],
                MET_phi[i],
            )
    
            if feats is None:
                continue
    
            X_aux1.append(feats)
    
    X_aux2 = np.asarray(X_aux1, dtype=np.float32)
    
    
    print('# of events available:          ', len(X_aux2))
    
    end_time = time.time()
    duracion_h5 = end_time - start_time
    print(f"Time processing the .h5:         {duracion_h5:.2f} s")
    
    
    
    
    # Create the DataFrame
    df = pd.DataFrame(X_aux2, columns=feats_label)
    
    # Save to CSV
    output_folder = DataFolder + "signal/data_df/"
    
    filename = "df_" + BP_sel + ".csv"
    df.to_csv(output_folder + filename, index=False)
    
    print(f"File with features saved:     {filename}")

    print("\n--------------------------------------------------\n")

BP2381
# of events available:           6850
Time processing the .h5:         0.83 s
File with features saved:     df_BP2381.csv

--------------------------------------------------



In [9]:
BPs_selected = ["BP35018"]


for BP_sel in BPs_selected:
    
    start_time = time.time()

    
    print(BP_sel)

        
    fname = DataFolder + "signal/" + BP_sel + "_cuts2.h5"
        
    X_aux1 = []
    
    with h5py.File(fname, "r") as f:
        # Load everything first (RAM should be ok)
        gam_pt = f["photon_pt"][:]
        gam_eta = f["photon_eta"][:]
        gam_phi = f["photon_phi"][:]
        btag_pt = f["btag_pt"][:]
        btag_eta = f["btag_eta"][:]
        btag_phi = f["btag_phi"][:]
        jet_pt = f["jet_pt"][:]
        jet_eta = f["jet_eta"][:]
        jet_phi = f["jet_phi"][:]
        MET_pt = f["MET_pt"][:]
        MET_phi = f["MET_phi"][:]
    
    
        for i in range(len(gam_pt)):
            feats = build_features_event(
                gam_pt[i],
                gam_eta[i],
                gam_phi[i],
                btag_pt[i],
                btag_eta[i],
                btag_phi[i],
                jet_pt[i],
                jet_eta[i],
                jet_phi[i],
                MET_pt[i],
                MET_phi[i],
            )
    
            if feats is None:
                continue
    
            X_aux1.append(feats)
    
    X_aux2 = np.asarray(X_aux1, dtype=np.float32)
    
    
    print('# of events available:          ', len(X_aux2))
    
    end_time = time.time()
    duracion_h5 = end_time - start_time
    print(f"Time processing the .h5:         {duracion_h5:.2f} s")
    
    
    
    
    # Create the DataFrame
    df = pd.DataFrame(X_aux2, columns=feats_label)
    
    # Save to CSV
    output_folder = DataFolder + "signal/data_df/"
    
    filename = "df_" + BP_sel + ".csv"
    df.to_csv(output_folder + filename, index=False)
    
    print(f"File with features saved:     {filename}")

    print("\n--------------------------------------------------\n")

BP35018
# of events available:           7064
Time processing the .h5:         0.88 s
File with features saved:     df_BP35018.csv

--------------------------------------------------



## LOAD

### Background

In [12]:
CHs_selected = ["bbaa", "tth", "zh"]


df_B_all = {}
X_B_all = {}


output_folder = DataFolder + "backgrounds/data_df/"

for CH_sel in CHs_selected:
    # Read the file
    file_path = output_folder + "df_" + CH_sel + ".csv"
    
    df_temp = pd.read_csv(file_path)

    # get the features
    feats_label = [c for c in df_temp.columns]
    
    # Save in the dictionary
    df_B_all[CH_sel] = df_temp
    X_B_all[CH_sel] = df_temp[feats_label].values
    
    # Print
    print(f"X_{CH_sel}.shape: {X_B_all[CH_sel].shape}")

X_bbaa.shape: (67056, 26)
X_tth.shape: (154321, 26)
X_zh.shape: (57657, 26)


In [13]:
df_B_all['bbaa']

,num_photons,num_jets,num_btag,MET_pT,MET_phi,photon1_pT,photon1_eta,photon1_phi,photon2_pT,photon2_eta,...,btag2_phi,m_hh,m_phopho,m_bb,pT_hh,pT_phopho,pT_bb,dR_hh,dR_phopho,dR_bb
0,2.0,1.0,2.0,16.25,1.116,137.77,0.076,0.507,95.61,-0.426,...,1.429,311.28302,100.338770,73.241790,170.384830,226.30931,90.48887,2.743451,0.884301,1.968216
1,2.0,2.0,2.0,38.86,3.063,262.94,-1.678,0.729,42.80,-2.139,...,2.310,795.99280,136.369350,86.347070,306.783800,301.87305,171.61840,2.812181,1.365191,0.928698
2,2.0,3.0,2.0,15.28,1.342,139.60,1.567,-2.288,84.58,1.065,...,2.171,228.97836,82.447014,19.830896,232.930040,217.58478,86.18144,1.622653,0.761044,0.445702
3,2.0,0.0,2.0,17.29,-1.617,124.66,-0.282,-1.187,54.13,-0.428,...,2.578,342.38983,69.826010,40.837986,8.665487,178.38802,161.96883,3.111648,0.876249,0.538283
4,2.0,1.0,2.0,39.08,0.408,202.10,1.193,-2.879,90.42,1.271,...,0.793,595.13257,134.399800,65.212975,46.995247,292.33000,211.52106,3.221903,1.039929,0.747819
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67051,2.0,0.0,2.0,23.37,1.023,95.27,1.600,-2.009,67.28,0.849,...,-0.565,348.95060,126.532480,125.107574,16.089449,151.57219,166.24295,3.058079,1.699159,1.711800
67052,2.0,3.0,2.0,22.11,-2.295,75.28,0.410,0.136,40.96,1.740,...,2.115,396.03232,81.377540,153.974440,76.976875,93.89072,215.93437,3.067214,1.368189,1.873034
67053,2.0,1.0,2.0,23.45,-1.861,67.58,-1.161,1.312,55.31,-0.198,...,-1.568,261.19660,89.528060,52.758488,26.750349,109.06569,82.42669,2.978603,1.483156,1.104240
67054,2.0,0.0,2.0,25.15,-1.663,101.13,2.150,1.498,36.38,2.056,...,3.012,296.41504,69.232640,124.723470,19.831417,137.39183,149.44418,3.140787,1.213646,1.950153


### Signal

In [14]:
BPs_selected = ["BP1", "BP28", "BP62", "BP63",
                "BP219", "BP220", "BP586", "BP635",
                "BP673", "BP684", "BP685", "BP728", "BP754"]


df_S_all = {}
X_S_all = {}


output_folder = DataFolder + "signal/data_df/"

for BP_sel in BPs_selected:
    # Read the file
    file_path = output_folder + "df_" + BP_sel + ".csv"
    
    df_temp = pd.read_csv(file_path)

    # get the features
    feats_label = [c for c in df_temp.columns]
    
    # Save in the dictionary
    df_S_all[BP_sel] = df_temp
    X_S_all[BP_sel] = df_temp[feats_label].values
    
    # Print
    print(f"X_{BP_sel}.shape: {X_S_all[BP_sel].shape}")

X_BP1.shape: (153551, 26)
X_BP28.shape: (7300, 26)
X_BP62.shape: (7736, 26)
X_BP63.shape: (8062, 26)
X_BP219.shape: (7590, 26)
X_BP220.shape: (11096, 26)
X_BP586.shape: (11057, 26)
X_BP635.shape: (7258, 26)
X_BP673.shape: (3708, 26)
X_BP684.shape: (3662, 26)
X_BP685.shape: (3439, 26)
X_BP728.shape: (3387, 26)
X_BP754.shape: (3691, 26)


In [15]:
df_S_all["BP28"]

,num_photons,num_jets,num_btag,MET_pT,MET_phi,photon1_pT,photon1_eta,photon1_phi,photon2_pT,photon2_eta,...,btag2_phi,m_hh,m_phopho,m_bb,pT_hh,pT_phopho,pT_bb,dR_hh,dR_phopho,dR_bb
0,2.0,1.0,2.0,22.80,1.782,179.13,0.499,-0.723,47.02,-0.767,...,2.834,327.89526,124.873440,18.117577,144.606020,198.37772,84.22343,3.076976,1.274973,0.436079
1,2.0,0.0,2.0,14.11,2.300,178.70,0.155,-0.452,159.59,0.685,...,2.339,807.67820,121.564995,140.000440,20.166890,326.51953,371.93353,3.252479,0.718418,0.854285
2,2.0,0.0,2.0,33.57,-0.698,118.07,-0.239,-0.689,47.39,0.538,...,0.235,343.38217,127.521194,63.348100,172.089770,155.45260,206.10179,2.099927,1.875521,0.895416
3,2.0,2.0,2.0,20.02,-1.033,165.46,-1.127,2.363,44.62,-1.767,...,0.430,406.79974,127.186330,110.233110,52.019108,203.00598,163.95506,2.909731,1.588791,1.337778
4,2.0,2.0,2.0,48.19,-2.036,228.93,-1.179,1.012,129.08,-1.349,...,-1.965,708.54630,127.409195,58.478760,91.557060,356.81815,249.54417,3.348543,0.757327,0.571995
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7295,2.0,1.0,2.0,30.14,1.771,612.54,-0.642,-2.183,89.64,-1.162,...,-2.805,481.78290,128.326000,122.802180,776.270100,691.76666,124.99940,1.175780,0.542041,1.584690
7296,2.0,3.0,2.0,29.94,-1.231,199.31,-0.273,1.512,164.71,-0.592,...,-1.189,705.44430,123.635460,91.783700,47.160630,359.44144,300.88794,3.100547,0.690149,0.647074
7297,2.0,1.0,3.0,50.96,-1.987,380.85,-0.033,1.127,208.64,0.388,...,-2.679,961.57120,125.713020,162.211210,270.196200,577.59985,340.04860,2.818551,0.443041,1.336465
7298,2.0,2.0,2.0,43.82,-0.693,123.54,0.374,2.162,73.49,0.551,...,-1.222,455.81772,125.517100,110.592660,72.591070,196.30876,236.90143,3.063466,1.433157,0.999082


In [16]:
X_S_all["BP28"].shape

(7300, 26)